# ScamSense — Notebook 06: LangGraph 3-Agent Pipeline

Implements a three-agent LangGraph pipeline — **100% free, no paid APIs required**.

| Agent | Role | Method |
|---|---|---|
| Detection Agent | Classify scam/ham + confidence | XLM-RoBERTa (HuggingFace) |
| Explanation Agent | Feature attribution — why is it a scam? | SHAP on classifier logits |
| Risk Agent | Severity score + SPF taxonomy label | Rule-based SPF 2025 taxonomy |

**Input**: raw message string  
**Output**: `{label, confidence, language, top_features, scam_type, risk_score, risk_tier, advice}`

## Cell 1 — Install dependencies

In [ ]:
# ── Install (no version pins — let Kaggle resolve) ─────────────────────────
!pip install -q langgraph langdetect shap

# Restart runtime after install
import os
os.kill(os.getpid(), 9)

## Cell 2 — Imports & setup

In [1]:
import re, json
from pathlib import Path
from typing import TypedDict, Optional
from collections import Counter

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import shap
from langdetect import detect, LangDetectException
from langgraph.graph import StateGraph, END

# ── Kaggle Secrets ──────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets  = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None  # public model, token optional

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = Path("/kaggle/working")
print(f"Device : {DEVICE}")
print(f"Base   : {BASE_DIR}")

Device : cuda
Base   : /kaggle/working


## Cell 3 — Load XLM-RoBERTa classifier

In [2]:
HF_MODEL_ID   = "Bhoovika/scamsense-xlmroberta"
clf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
clf_model     = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_ID, token=HF_TOKEN
).to(DEVICE)
clf_model.eval()
LABEL_MAP = {0: "ham", 1: "scam"}

def run_classifier(text: str) -> dict:
    inputs = clf_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=128, padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs    = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_idx = int(probs.argmax())
    return {
        "label"      : LABEL_MAP[pred_idx],
        "confidence" : round(float(probs[pred_idx]), 4),
        "scam_prob"  : round(float(probs[1]), 4),
        "logits"     : logits.cpu().numpy()[0].tolist(),
    }

print("Classifier check:", run_classifier("You won $10,000! Claim now."))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier check: {'label': 'scam', 'confidence': 0.9999, 'scam_prob': 0.9999, 'logits': [-4.8981170654296875, 4.316030979156494]}


## Cell 4 — SPF 2025 Risk Taxonomy (rule-based, no API needed)

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SPF 2025 Annual Scam Brief taxonomy
# Source: Singapore Police Force, Annual Scams & Cybercrime Brief 2025
# Top 5 by cases: e-commerce, phishing, job, investment, govt impersonation
# Top 5 by losses: investment ($336.2M), govt impersonation ($242.9M),
#                  job ($123.5M), phishing ($39.9M), BEC ($35.3M)
# ─────────────────────────────────────────────────────────────────────────────

SPF_TAXONOMY = {
    "investment": {
        "spf_name"    : "Investment Scam",
        "2025_cases"  : 5462,
        "2025_losses" : 336.2,   # SGD millions
        "avg_loss"    : 61559,   # SGD per victim
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 95,
        "keywords"    : [
            r"invest", r"return", r"profit", r"forex", r"crypto",
            r"trading", r"guaranteed", r"passive income", r"portfolio",
            r"capital", r"dividend", r"scheme", r"fund", r"roi",
            r"bitcoin", r"ethereum", r"tether", r"usdt", r"wallet",
            r"keuntungan", r"pelaburan",           # Malay
            r"முதலீட்", r"லாபம்",                  # Tamil
            r"投资", r"收益", r"理财", r"赚钱",      # Mandarin
        ],
        "advice"      : (
            "Investment scams caused the HIGHEST losses in Singapore in 2025 — "
            "$336.2 million (SPF 2025). Never transfer money to strangers for "
            "'investments'. Verify any investment opportunity at MAS "
            "(mas.gov.sg/investor-alert-list). Report to SPF at 1800-255-0000."
        ),
    },
    "impersonation": {
        "spf_name"    : "Government Officials Impersonation Scam",
        "2025_cases"  : 3363,
        "2025_losses" : 242.9,
        "avg_loss"    : 72229,
        "risk_tier"   : "CRITICAL",
        "risk_score"  : 93,
        "keywords"    : [
            r"police", r"spf", r"cpf", r"ica", r"mas", r"iras", r"hdb",
            r"officer", r"detective", r"warrant", r"arrest", r"investigation",
            r"money laundering", r"safety account", r"court", r"ministry",
            r"government", r"official", r"authority", r"polis",  # Malay
            r"警察", r"公安", r"调查", r"安全账户",              # Mandarin
            r"காவல்", r"அரசு",                               # Tamil
        ],
        "advice"      : (
            "Cases MORE THAN DOUBLED in 2025 (+123.6%), with $242.9M lost (SPF 2025). "
            "Singapore government officials will NEVER ask you to transfer money, "
            "disclose banking details, or install unofficial apps over a phone call. "
            "Hang up immediately and call ScamShield at 1799 to verify."
        ),
    },
    "job": {
        "spf_name"    : "Job Scam",
        "2025_cases"  : 5575,
        "2025_losses" : 123.5,
        "avg_loss"    : 22163,
        "risk_tier"   : "HIGH",
        "risk_score"  : 80,
        "keywords"    : [
            r"job", r"work from home", r"part.?time", r"hiring", r"salary",
            r"task", r"commission", r"earn", r"vacancy", r"recruit",
            r"registration fee", r"training fee", r"deposit", r"agent fee",
            r"kerja", r"gaji", r"pendapatan",       # Malay
            r"வேலை", r"சம்பளம்",                   # Tamil
            r"工作", r"兼职", r"佣金", r"招聘",     # Mandarin
        ],
        "advice"      : (
            "Job scams cost Singaporeans $123.5M in 2025 (SPF 2025). "
            "Legitimate employers never ask for upfront fees before employment. "
            "Verify job offers at mom.gov.sg. Report to MOM or call 1800-255-0000."
        ),
    },
    "phishing": {
        "spf_name"    : "Phishing Scam",
        "2025_cases"  : 6264,
        "2025_losses" : 39.9,
        "avg_loss"    : 6384,
        "risk_tier"   : "HIGH",
        "risk_score"  : 78,
        "keywords"    : [
            r"click", r"link", r"verify", r"suspend", r"account", r"login",
            r"password", r"otp", r"credential", r"bank", r"dbs", r"ocbc",
            r"uob", r"posb", r"singpass", r"cpf", r"paynow", r"card",
            r"http", r"www\.", r"\.xyz", r"\.top", r"\.club",
            r"kata laluan", r"akaun",               # Malay
            r"கணக்கு", r"கடவுச்சொல்",              # Tamil
            r"账户", r"密码", r"验证", r"点击",      # Mandarin
        ],
        "advice"      : (
            "Phishing is the 2nd most common scam in Singapore (6,264 cases, SPF 2025). "
            "Never click unsolicited links or enter your OTP/password on any site "
            "from an SMS or email. Go directly to your bank's official website. "
            "Report phishing links to report@scamalert.sg."
        ),
    },
    "ecommerce": {
        "spf_name"    : "E-commerce Scam",
        "2025_cases"  : 6703,
        "2025_losses" : 16.7,
        "avg_loss"    : 2503,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 60,
        "keywords"    : [
            r"sell", r"selling", r"buy", r"cheap", r"deal", r"item",
            r"carousell", r"shopee", r"lazada", r"facebook marketplace",
            r"ship", r"delivery", r"transfer first", r"paynow first",
            r"deposit", r"pokemon", r"brand new", r"legit",
            r"jual", r"beli", r"murah",             # Malay
            r"விற்பனை", r"வாங்க",                  # Tamil
            r"出售", r"购买", r"便宜", r"转账",     # Mandarin
        ],
        "advice"      : (
            "E-commerce scams are the most common scam type in Singapore (6,703 cases, "
            "SPF 2025). Never PayNow before receiving goods. Meet in person for "
            "high-value items or use Carousell's protected payment. "
            "Report at go.gov.sg/scamalert."
        ),
    },
    "love": {
        "spf_name"    : "Internet Love Scam",
        "2025_cases"  : 917,
        "2025_losses" : 24.9,
        "avg_loss"    : 27202,
        "risk_tier"   : "HIGH",
        "risk_score"  : 75,
        "keywords"    : [
            r"love", r"relationship", r"dating", r"girlfriend", r"boyfriend",
            r"miss you", r"together", r"meet", r"lonely", r"soulmate",
            r"overseas", r"stranded", r"emergency", r"send money",
            r"cinta", r"sayang",                    # Malay
            r"அன்பு", r"காதல்",                    # Tamil
            r"爱", r"恋爱", r"想你",               # Mandarin
        ],
        "advice"      : (
            "Internet love scams averaged $27,202 per victim in 2025 (SPF 2025). "
            "Be wary of online relationships with people who never video call, "
            "claim to be overseas, and eventually ask for money. "
            "Talk to someone you trust and call ScamShield at 1799."
        ),
    },
    "unknown": {
        "spf_name"    : "Other Scam",
        "2025_cases"  : 9941,
        "2025_losses" : 135.1,
        "avg_loss"    : 13590,
        "risk_tier"   : "MEDIUM",
        "risk_score"  : 55,
        "keywords"    : [],
        "advice"      : (
            "This message shows scam indicators. Do not send money, share personal "
            "details or OTPs, or click unknown links. "
            "Verify via ScamShield app or call 1799 if unsure."
        ),
    },
}

def classify_scam_type(text: str) -> tuple[str, int, str, str]:
    """Match text against SPF taxonomy. Returns (scam_type, risk_score, risk_tier, advice)."""
    text_lower = text.lower()
    scores = {}
    for stype, info in SPF_TAXONOMY.items():
        if stype == "unknown":
            continue
        hits = sum(
            1 for kw in info["keywords"]
            if re.search(kw, text_lower)
        )
        if hits > 0:
            scores[stype] = hits
    if not scores:
        t = SPF_TAXONOMY["unknown"]
        return "unknown", t["risk_score"], t["risk_tier"], t["advice"]
    best = max(scores, key=scores.get)
    t = SPF_TAXONOMY[best]
    return best, t["risk_score"], t["risk_tier"], t["advice"]

# Quick test
print(classify_scam_type("guaranteed 300% returns on crypto investment!"))

('investment', 95, 'CRITICAL', "Investment scams caused the HIGHEST losses in Singapore in 2025 — $336.2 million (SPF 2025). Never transfer money to strangers for 'investments'. Verify any investment opportunity at MAS (mas.gov.sg/investor-alert-list). Report to SPF at 1800-255-0000.")


## Cell 5 — SHAP Explanation setup

In [4]:
# SHAP TextMasker uses the tokenizer to create perturbations
# PartitionExplainer is recommended for transformer models

def shap_predict(texts):
    """Wrapper for SHAP: returns scam probability for each text."""
    if isinstance(texts, str):
        texts = [texts]
    inputs = clf_tokenizer(
        list(texts),
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs  # shape (N, 2) — col 0=ham, col 1=scam

# Masker uses the tokenizer vocabulary
masker  = shap.maskers.Text(clf_tokenizer)
explainer = shap.Explainer(shap_predict, masker, output_names=["ham", "scam"])

def get_top_shap_features(text: str, top_n: int = 5) -> list[dict]:
    """
    Run SHAP on a single text.
    Returns top_n tokens most responsible for the scam prediction.
    """
    shap_values = explainer([text])
    # shap_values.values shape: (1, n_tokens, 2)
    # We want scam class (index 1)
    token_names  = shap_values.data[0]          # list of token strings
    token_shaps  = shap_values.values[0, :, 1]  # scam-class shap values

    pairs = [
        {"token": tok, "shap_value": round(float(val), 4)}
        for tok, val in zip(token_names, token_shaps)
        if tok not in ["", "▁", "<s>", "</s>", "<pad>"]
    ]
    # Sort by absolute SHAP, return top_n positive contributors
    pairs_sorted = sorted(pairs, key=lambda x: x["shap_value"], reverse=True)
    return pairs_sorted[:top_n]

# Quick test
print(get_top_shap_features("Your account has been suspended. Verify now."))

[{'token': 'ify ', 'shap_value': 0.3011}, {'token': 'now', 'shap_value': 0.2433}, {'token': 'suspend', 'shap_value': 0.2366}, {'token': 'account ', 'shap_value': 0.1925}, {'token': 'has ', 'shap_value': 0.0521}]


## Cell 6 — Language detector + LangGraph state schema

In [5]:
# ── Singlish heuristics ──────────────────────────────────────────────────────
SINGLISH_MARKERS = [
    r"\blah\b", r"\bleh\b", r"\bsia\b", r"\blor\b", r"\bwah\b",
    r"\baiyo\b", r"\bsiao\b", r"\bdun\b", r"\bwan\b", r"\bcan or not\b",
    r"\bgot meh\b", r"\bcan leh\b", r"\bone\b",
]
_singlish_re = re.compile("|".join(SINGLISH_MARKERS), re.IGNORECASE)

def detect_language(text: str) -> str:
    hits = len(_singlish_re.findall(text))
    words = len(text.split())
    if words > 0 and (hits / words) >= 0.08:
        return "singlish"
    try:
        d = detect(text)
        return {"en": "en", "ms": "ms", "id": "ms",
                "ta": "ta", "zh-cn": "zh", "zh-tw": "zh", "zh": "zh"}.get(d, "en")
    except LangDetectException:
        return "en"


# ── LangGraph state ───────────────────────────────────────────────────────────
class AgentState(TypedDict):
    # Input
    message      : str
    # Detection Agent outputs
    language     : Optional[str]
    label        : Optional[str]
    confidence   : Optional[float]
    scam_prob    : Optional[float]
    # Explanation Agent outputs
    top_features : Optional[list]
    # Risk Agent outputs
    scam_type    : Optional[str]
    spf_name     : Optional[str]
    risk_score   : Optional[int]
    risk_tier    : Optional[str]
    advice       : Optional[str]
    # Final
    output       : Optional[dict]

print("State schema defined ✅")

State schema defined ✅


## Cell 7 — Define the 3 agents as LangGraph nodes

In [7]:
# ── Agent 1: Detection ────────────────────────────────────────────────────────
def detection_agent(state: AgentState) -> AgentState:
    """Detects language and classifies scam/ham using XLM-RoBERTa."""
    lang   = detect_language(state["message"])
    result = run_classifier(state["message"])
    return {
        **state,
        "language"  : lang,
        "label"     : result["label"],
        "confidence": result["confidence"],
        "scam_prob" : result["scam_prob"],
    }


# ── Routing: skip explanation + risk if ham with high confidence ──────────────
def route_after_detection(state: AgentState) -> str:
    if state["label"] == "ham" and state["confidence"] >= 0.80:
        return "safe_exit"
    return "explanation_agent"


# ── Agent 2: Explanation (SHAP) ───────────────────────────────────────────────
def explanation_agent(state: AgentState) -> AgentState:
    """Uses SHAP to identify which tokens contributed most to the scam prediction."""
    top_features = get_top_shap_features(state["message"], top_n=5)
    return {**state, "top_features": top_features}


# ── Agent 3: Risk Scoring (SPF taxonomy) ──────────────────────────────────────
def risk_agent(state: AgentState) -> AgentState:
    """Maps message to SPF 2025 scam taxonomy and computes risk score."""
    scam_type, base_score, risk_tier, advice = classify_scam_type(state["message"])

    # Adjust risk score by model confidence
    # High confidence scam → full score; borderline → scale down
    adjusted_score = int(base_score * state["scam_prob"])
    adjusted_score = max(10, min(100, adjusted_score))

    # Re-tier based on adjusted score
    if adjusted_score >= 85:
        risk_tier = "CRITICAL"
    elif adjusted_score >= 65:
        risk_tier = "HIGH"
    elif adjusted_score >= 40:
        risk_tier = "MEDIUM"
    else:
        risk_tier = "LOW"

    spf_info = SPF_TAXONOMY.get(scam_type, SPF_TAXONOMY["unknown"])

    output = {
        "message"     : state["message"],
        "language"    : state["language"],
        "label"       : state["label"],
        "confidence"  : state["confidence"],
        "scam_prob"   : state["scam_prob"],
        "top_features": state.get("top_features", []),
        "scam_type"   : scam_type,
        "spf_name"    : spf_info["spf_name"],
        "spf_cases_2025"  : spf_info["2025_cases"],
        "spf_losses_2025" : spf_info["2025_losses"],
        "risk_score"  : adjusted_score,
        "risk_tier"   : risk_tier,
        "advice"      : advice,
    }
    return {**state, "scam_type": scam_type, "spf_name": spf_info["spf_name"],
            "risk_score": adjusted_score, "risk_tier": risk_tier,
            "advice": advice, "output": output}


# ── Safe exit for ham messages ────────────────────────────────────────────────
def safe_exit_node(state: AgentState) -> AgentState:
    output = {
        "message"     : state["message"],
        "language"    : state["language"],
        "label"       : "ham",
        "confidence"  : state["confidence"],
        "scam_prob"   : state["scam_prob"],
        "top_features": [],
        "scam_type"   : None,
        "spf_name"    : None,
        "risk_score"  : 0,
        "risk_tier"   : "NONE",
        "advice"      : "This message appears legitimate. No action needed.",
    }
    return {**state, "output": output}

print("All 3 agents defined ✅")

All 3 agents defined ✅


## Cell 8 — Assemble LangGraph state machine

In [8]:
graph = StateGraph(AgentState)

# Register nodes
graph.add_node("detection_agent",   detection_agent)
graph.add_node("explanation_agent", explanation_agent)
graph.add_node("risk_agent",        risk_agent)
graph.add_node("safe_exit",         safe_exit_node)

# Entry
graph.set_entry_point("detection_agent")

# Conditional routing after Detection Agent
graph.add_conditional_edges(
    "detection_agent",
    route_after_detection,
    {
        "safe_exit"        : "safe_exit",
        "explanation_agent": "explanation_agent",
    }
)

# Scam path: Explanation → Risk → END
graph.add_edge("explanation_agent", "risk_agent")
graph.add_edge("risk_agent",        END)
graph.add_edge("safe_exit",         END)

pipeline = graph.compile()
print("✅ LangGraph pipeline compiled")
print("Nodes:", list(pipeline.get_graph().nodes.keys()))

✅ LangGraph pipeline compiled
Nodes: ['__start__', 'detection_agent', 'explanation_agent', 'risk_agent', 'safe_exit', '__end__']


## Cell 9 — Helper: run and pretty-print

In [9]:
TIER_ICONS = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢", "NONE": "✅"}

def run_pipeline(message: str, verbose: bool = True) -> dict:
    initial: AgentState = {
        "message": message, "language": None, "label": None,
        "confidence": None, "scam_prob": None, "top_features": None,
        "scam_type": None, "spf_name": None, "risk_score": None,
        "risk_tier": None, "advice": None, "output": None,
    }
    final  = pipeline.invoke(initial)
    result = final["output"]

    if verbose:
        tier = result["risk_tier"]
        icon = TIER_ICONS.get(tier, "❓")
        print(f"{icon} [{tier}] {result['label'].upper()} — confidence: {result['confidence']:.1%} | lang: {result['language']}")
        if result["label"] == "scam":
            print(f"   SPF Type  : {result['spf_name']}")
            print(f"   Risk Score: {result['risk_score']}/100")
            print(f"   SPF 2025  : {result['spf_cases_2025']:,} cases | ${result['spf_losses_2025']}M lost")
            if result.get("top_features"):
                top_toks = ", ".join(
                    f"'{f['token']}'({f['shap_value']:+.3f})"
                    for f in result["top_features"]
                )
                print(f"   SHAP keys : {top_toks}")
            print(f"   Advice    : {result['advice']}")
        print("-" * 72)
    return result

print("Helper ready ✅")

Helper ready ✅


## Cell 10 — Inference: one per language + scam type

In [10]:
import pandas as pd

test_cases = [
    ("en / phishing",
     "Your POSB account has been suspended. Verify your details at http://posb-secure-login.xyz or it will be closed."),
    ("en / investment",
     "Exclusive crypto investment opportunity! Guaranteed 300% returns in 30 days. Join our Telegram group now."),
    ("ms / job scam",
     "Kerja mudah dari rumah! Pendapatan RM5,000 sebulan. Bayar yuran pendaftaran RM200 sahaja untuk mula."),
    ("ta / investment",
     "உங்கள் முதலீட்டில் 300% லாபம் உறுதி. இப்போதே பதிவு செய்யுங்கள், வாய்ப்பு வரையறுக்கப்பட்டது!"),
    ("zh / impersonation",
     "您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户，否则将被逮捕。"),
    ("singlish / ecommerce",
     "Eh bro selling iPhone 15 Pro Max $400 only lah! Very cheap one. PayNow me first then I ship lor. Confirm legit wan!"),
    ("en / ham",
     "Hi! Reminder: your dentist appointment is tomorrow at 2pm at Raffles Dental. Please reply to confirm."),
]

print("=" * 72)
print("SCAMSENSE 3-AGENT PIPELINE — INFERENCE RESULTS")
print("=" * 72)

all_results = []
for name, msg in test_cases:
    print(f"\n▶ {name}")
    print(f"  {msg[:90]}...")
    r = run_pipeline(msg, verbose=True)
    r["test_case"] = name
    all_results.append(r)

print("\n✅ All inference examples complete")

SCAMSENSE 3-AGENT PIPELINE — INFERENCE RESULTS

▶ en / phishing
  Your POSB account has been suspended. Verify your details at http://posb-secure-login.xyz ...
🟠 [HIGH] SCAM — confidence: 100.0% | lang: en
   SPF Type  : Phishing Scam
   Risk Score: 78/100
   SPF 2025  : 6,264 cases | $39.9M lost
   SHAP keys : 'suspend'(+0.210), 'account '(+0.209), 'closed'(+0.186), 'POS'(+0.163), '.'(+0.073)
   Advice    : Phishing is the 2nd most common scam in Singapore (6,264 cases, SPF 2025). Never click unsolicited links or enter your OTP/password on any site from an SMS or email. Go directly to your bank's official website. Report phishing links to report@scamalert.sg.
------------------------------------------------------------------------

▶ en / investment
  Exclusive crypto investment opportunity! Guaranteed 300% returns in 30 days. Join our Tele...
🔴 [CRITICAL] SCAM — confidence: 100.0% | lang: en
   SPF Type  : Investment Scam
   Risk Score: 95/100
   SPF 2025  : 5,462 cases | $336.2M los

## Cell 11 — Summary table + save

In [11]:
rows = []
for r in all_results:
    top_tok = r["top_features"][0]["token"] if r.get("top_features") else "-"
    rows.append({
        "test_case"  : r["test_case"],
        "label"      : r["label"],
        "confidence" : f"{r['confidence']:.1%}",
        "language"   : r["language"],
        "spf_type"   : r.get("spf_name") or "—",
        "risk_tier"  : r["risk_tier"],
        "risk_score" : r["risk_score"],
        "top_token"  : top_tok,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

out_dir = BASE_DIR / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "06_agent_inference_results.csv", index=False)
print("\n✅ Results saved → results/06_agent_inference_results.csv")

           test_case label confidence language        spf_type risk_tier  risk_score top_token
       en / phishing  scam     100.0%       en   Phishing Scam      HIGH          78   suspend
     en / investment  scam     100.0%       en Investment Scam  CRITICAL          95       now
       ms / job scam  scam     100.0%       ms        Job Scam      HIGH          80    untuk 
     ta / investment  scam     100.0%       ta Investment Scam  CRITICAL          95 வாய்ப்பு 
  zh / impersonation   ham     100.0%       zh               —      NONE           0         -
singlish / ecommerce  scam      99.9% singlish E-commerce Scam    MEDIUM          59         $
            en / ham   ham      99.4%       en               —      NONE           0         -

✅ Results saved → results/06_agent_inference_results.csv


## Cell 12 — Save pipeline + push to GitHub

In [13]:
import subprocess, json

# Note: LangGraph compiled graphs cannot be pickled — standard LangGraph pattern.
# Save SPF taxonomy as JSON for use in downstream notebooks instead.
models_dir = BASE_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
taxonomy_out = {k: {ek: ev for ek, ev in v.items() if ek != "keywords"}
                for k, v in SPF_TAXONOMY.items()}
with open(models_dir / "spf_taxonomy.json", "w") as f:
    json.dump(taxonomy_out, f, indent=2)
print("✅ SPF taxonomy saved to models/spf_taxonomy.json")
print("\n" + "=" * 60)
print("✅ Notebook 06 complete!")
print("   Agent 1 : Detection (XLM-RoBERTa)")
print("   Agent 2 : Explanation (SHAP token attribution)")
print("   Agent 3 : Risk scoring (SPF 2025 taxonomy)")
print("   Next    : Notebook 07 — FAISS RAG over SPF advisories")
print("=" * 60)

✅ SPF taxonomy saved to models/spf_taxonomy.json

✅ Notebook 06 complete!
   Agent 1 : Detection (XLM-RoBERTa)
   Agent 2 : Explanation (SHAP token attribution)
   Agent 3 : Risk scoring (SPF 2025 taxonomy)
   Next    : Notebook 07 — FAISS RAG over SPF advisories
